In [0]:
catalog = "hetul_catalog_ott_02"
from pyspark.sql.functions import col, trim, lower, when

# Step 1: Load Bronze Table
bronze_users = spark.table(f"{catalog}.bronze.bronze_users_csv")

# Step 2: Clean Data
silver_users = (
    bronze_users
    # Trim whitespaces from all string columns
    .withColumn("user_id", trim(col("user_id")))
    .withColumn("plan_type", trim(col("plan_type")))
    .withColumn("billing_cycle", trim(col("billing_cycle")))
    .withColumn("payment_status", trim(col("payment_status")))
    .withColumn("churn_flag", trim(col("churn_flag")))

    # Drop invalid user_id
    .filter((col("user_id").isNotNull()) & (~lower(col("user_id")).startswith("invalid")))

    # Standardize plan_type and replace invalid/missing with 'unknown'
    .withColumn(
        "plan_type",
        when(lower(col("plan_type")).isin("basic","standard","premium"), col("plan_type"))
        .otherwise("unknown")
    )

    # Standardize billing_cycle
    .withColumn(
        "billing_cycle",
        when(lower(col("billing_cycle")).isin("monthly","yearly"), col("billing_cycle"))
        .otherwise("unknown")
    )

    # Standardize payment_status
    .withColumn(
        "payment_status",
        when(lower(col("payment_status")).isin("active","cancelled","delinquent"), lower(col("payment_status")))
        .otherwise("unknown")
    )

    # Standardize churn_flag: Active → 0, others → 1, missing/invalid → 0
    .withColumn(
        "churn_flag",
        when(lower(col("payment_status")) == "active", 0)
        .otherwise(1)
    )
)

# Step 3: Write Silver Table

silver_users_cleaned = silver_users.drop("ingest_file")

# Overwrite the table and remove column completely
silver_users_cleaned.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalog}.silver.silver_users")


In [0]:
silver_users_casted = silver_users.select(
    col("user_id").cast("string"),
    col("plan_type").cast("string"),
    col("billing_cycle").cast("string"),
    col("payment_status").cast("string"),
    col("churn_flag").cast("int")  # cast churn_flag to integer
)
silver_users_casted.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalog}.silver.silver_users")


In [0]:
%sql
select * from hetul_catalog_ott_02.silver.silver_users

user_id,plan_type,billing_cycle,payment_status,churn_flag
U00001,premium,monthly,active,0
U00002,basic,monthly,active,0
U00003,premium,monthly,cancelled,1
U00004,basic,monthly,active,0
U00005,basic,monthly,cancelled,1
U00006,unknown,yearly,active,0
U00007,premium,yearly,active,0
U00008,unknown,yearly,delinquent,1
U00009,basic,monthly,delinquent,1
U00010,basic,yearly,active,0


In [0]:
from pyspark.sql.functions import col, trim, when, lower

# Load bronze content table
bronze_content = spark.table(f"{catalog}.bronze.bronze_content_csv")

# Clean content data
silver_content = (
    bronze_content
    # Trim whitespaces
    .withColumn("content_id", trim(col("content_id")))
    .withColumn("title", trim(col("title")))
    .withColumn("genre", trim(col("genre")))
    .withColumn("duration_min", trim(col("duration_min")))
    .withColumn("region_availability", trim(col("region_availability")))

    # Drop invalid content_id (null or starts with "invalid")
    .filter((col("content_id").isNotNull()) & (~lower(col("content_id")).startswith("invalid")))

    # Replace invalid/null/blank title with 'unknown'
    .withColumn(
        "title",
        when(
            col("title").isNull() | (col("title") == "") | lower(col("title")).startswith("invalid"),
            "unknown"
        ).otherwise(col("title"))
    )

    # Replace invalid/null/blank genre with 'unknown'
    .withColumn(
        "genre",
        when(
            col("genre").isNull() | (col("genre") == "") | lower(col("genre")).startswith("invalid"),
            "unknown"
        ).otherwise(col("genre"))
    )

    # Replace null/blank/negative/invalid duration with 0, cast to int
    .withColumn(
        "duration_min",
        when(col("duration_min").rlike("^[0-9]+$"), col("duration_min").cast("int")).otherwise(0)
    )

    # Remove outliers: duration > 300 minutes
    .filter(col("duration_min") <= 300)

    # Replace invalid/null/blank region_availability with 'unknown'
    .withColumn(
        "region_availability",
        when(
            col("region_availability").isNull() | (col("region_availability") == "") | lower(col("region_availability")).startswith("invalid"),
            "unknown"
        ).otherwise(col("region_availability"))
    )

    # Drop the ingest_file column if exists
    .drop(*[c for c in ["ingest_file"] if c in bronze_content.columns])
)

# Write cleaned silver content table
silver_content.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalog}.silver.silver_content")


In [0]:
%sql
select * from hetul_catalog_ott_02.silver.silver_content

content_id,title,genre,duration_min,region_availability
C0001,Movie_S8HMB,Documentary,150,APAC
C0002,Movie_T4P50,Documentary,123,NA
C0003,Movie_OUHTC,Sci-Fi,145,APAC
C0004,Movie_3WVED,Action,116,unknown
C0005,Movie_1HSSB,Comedy,0,LATAM
C0006,Movie_SF6F7,unknown,71,LATAM
C0007,Movie_90KM3,Sci-Fi,155,APAC
C0008,Movie_RALDO,unknown,36,LATAM
C0009,Movie_JX8UI,Drama,179,unknown
C0010,Movie_GWW4F,Kids,92,NA


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, trim, when, avg, stddev, lower

catalog = "hetul_catalog_ott_02"

# Load bronze ads table
bronze_ads = spark.table(f"{catalog}.bronze.bronze_ads_csv")

# ----------------------------
# Trim all string columns
# ----------------------------
ads_trimmed = bronze_ads.select(
    trim(col("ad_id")).alias("ad_id"),
    trim(col("campaign")).alias("campaign"),
    trim(col("content_id")).alias("content_id"),
    trim(col("impressions")).alias("impressions"),
    trim(col("clicks")).alias("clicks"),
    trim(col("revenue")).alias("revenue")
)

# ----------------------------
# Remove rows with invalid/null/blank values
# ----------------------------
ads_cleaned = ads_trimmed.filter(
    (col("ad_id").isNotNull()) & (col("ad_id") != "") & (~lower(col("ad_id")).startswith("invalid")) &
    (col("campaign").isNotNull()) & (col("campaign") != "") & (~lower(col("campaign")).startswith("invalid")) 
).withColumn(
    "content_id", when(col("content_id").isNull() | (col("content_id") == "") | lower(col("content_id")).startswith("invalid"),
                       "unknown").otherwise(col("content_id"))
).withColumn(
    "revenue", when(col("revenue").rlike(r"^[0-9]+(\.[0-9]+)?$"), col("revenue").cast("float")).otherwise(0)
).withColumn(
    "impressions", when(col("impressions").rlike(r"^[0-9]+$") & (col("impressions").cast("int") >= 0),
                        col("impressions").cast("int")).otherwise(0)
).withColumn(
    "clicks", when(col("clicks").rlike(r"^[0-9]+$") & (col("clicks").cast("int") >= 0),
                    col("clicks").cast("int")).otherwise(0)
)

# Ensure clicks <= impressions
ads_cleaned = ads_cleaned.withColumn(
    "clicks", when(col("clicks") > col("impressions"), col("impressions")).otherwise(col("clicks"))
)

# Compute averages and standard deviations for outlier handling
impr_avg, click_avg = ads_cleaned.select(avg(col("impressions")), avg(col("clicks"))).first()
impr_std, click_std = ads_cleaned.select(stddev(col("impressions")), stddev(col("clicks"))).first()

# Replace outliers with mean if they deviate > 3*std
ads_final = ads_cleaned.withColumn(
    "impressions", when(F.abs(col("impressions") - impr_avg) > 3*impr_std, impr_avg).otherwise(col("impressions"))
).withColumn(
    "clicks", when(F.abs(col("clicks") - click_avg) > 3*click_std, click_avg).otherwise(col("clicks"))
)

# Drop ingest_file if exists
if "ingest_file" in ads_final.columns:
    ads_final = ads_final.drop("ingest_file")

# Write to Silver table
ads_final.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalog}.silver.silver_ads")


In [0]:
%sql
select * from hetul_catalog_ott_02.silver.silver_ads

ad_id,campaign,content_id,impressions,clicks,revenue
A00001,CAMP_B,C0091,366.0,127.0,23.51
A00002,CAMP_A,C0099,861.0,191.0,42.44
A00003,CAMP_B,C0057,760.0,51.0,18.81
A00004,CAMP_A,C0059,409.0,160.0,5.84
A00006,CAMP_A,C0039,994.0,14.0,22.89
A00007,CAMP_B,C0060,0.0,0.0,40.07
A00009,CAMP_B,C0017,707.0,183.0,23.35
A00010,CAMP_A,C0077,624.0,68.0,28.05
A00011,CAMP_A,C0010,761.0,48.0,36.93
A00012,CAMP_A,C0056,928.0,33.0,26.61


In [0]:
# Base paths in ADLS
base_path        = "abfss://rawott@hetulstorage.dfs.core.windows.net/raw_data"
schema_base_path = "abfss://rawott@hetulstorage.dfs.core.windows.net/schemas"
chkpt_base_path  = "abfss://rawott@hetulstorage.dfs.core.windows.net/chkpts"

from pyspark.sql.functions import col
from pyspark.sql.types import *

# ----------------------------
# Events CSV - streaming with trigger once
# ----------------------------

# Define schema for Events (all as StringType in Bronze)
events_schema = StructType([
    StructField("event_id", StringType(), True),
    StructField("user_id", StringType(), True),
    StructField("content_id", StringType(), True),
    StructField("start_time", StringType(), True),
    StructField("end_time", StringType(), True),
    StructField("device", StringType(), True),
    StructField("geo", StringType(), True),
    StructField("bitrate", StringType(), True),
    StructField("errors", StringType(), True)
])

bronze_events_csv = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.schemaLocation", f"{schema_base_path}/events_csv")
    .schema(events_schema)
    .load(f"{base_path}/events_data")
    .withColumn("ingest_file", col("_metadata.file_path"))
)

query_events_csv = (bronze_events_csv.writeStream
    .format("delta")
    .option("checkpointLocation", f"{chkpt_base_path}/events_csv")
    .outputMode("append")
    .trigger(once=True)
    .toTable(f"{catalog}.bronze.bronze_events_csv")
)


In [0]:
%sql
select * from hetul_catalog_ott_02.bronze.bronze_events_csv

event_id,user_id,content_id,start_time,end_time,device,geo,bitrate,errors,ingest_file
E000001,INVALID_9NY3,C0031,2025-08-28T14:02:07.064571,2025-08-28T14:27:07.064571,Web,DE,1080.0,network_drop,abfss://rawott@hetulstorage.dfs.core.windows.net/raw_data/events_data/events2.csv
E000002,U00027,C0035,2025-08-28T23:20:07.064571,2025-08-29T00:48:07.064571,Web,UK,1440.0,network_drop,abfss://rawott@hetulstorage.dfs.core.windows.net/raw_data/events_data/events2.csv
E000003,U00030,C0044,2025-08-29T02:03:07.064571,2025-08-29T02:25:07.064571,Mobile,DE,240.0,buffering,abfss://rawott@hetulstorage.dfs.core.windows.net/raw_data/events_data/events2.csv
E000004,U00212,C0059,2025-08-28T19:21:07.064571,2025-08-28T19:30:07.064571,Tablet,IN,null,crash,abfss://rawott@hetulstorage.dfs.core.windows.net/raw_data/events_data/events2.csv
E000005,U00306,C0080,2025-08-28T15:07:07.064571,2025-08-28T16:06:07.064571,Web,DE,1080.0,null,abfss://rawott@hetulstorage.dfs.core.windows.net/raw_data/events_data/events2.csv
E000006,U00442,C0002,2025-08-29T04:55:07.064571,2025-08-29T05:10:07.064571,Tablet,DE,2160.0,INVALID_CVJX,abfss://rawott@hetulstorage.dfs.core.windows.net/raw_data/events_data/events2.csv
E000007,U00312,C0059,2025-08-29T03:53:07.064571,2025-08-29T04:04:07.064571,Web,DE,360.0,network_drop,abfss://rawott@hetulstorage.dfs.core.windows.net/raw_data/events_data/events2.csv
E000008,U00155,C0005,INVALID_LBU9,2025-08-28T14:27:07.064571,SmartTV,BR,2160.0,null,abfss://rawott@hetulstorage.dfs.core.windows.net/raw_data/events_data/events2.csv
E000009,null,C0051,2025-08-29T00:44:07.064571,2025-08-29T01:01:07.064571,Tablet,US,720.0,null,abfss://rawott@hetulstorage.dfs.core.windows.net/raw_data/events_data/events2.csv
E000010,U00338,C0048,2025-08-28T17:28:07.064571,2025-08-28T18:22:07.064571,Tablet,JP,1080.0,network_drop,abfss://rawott@hetulstorage.dfs.core.windows.net/raw_data/events_data/events2.csv


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, trim, when, to_timestamp, lower, lit, coalesce

catalog = "hetul_catalog_ott_02"
bronze = spark.table(f"{catalog}.bronze.bronze_events_csv")

ALLOWED_BITRATES = [240, 360, 480, 720, 1080, 2160]

# ----------------------------
# 1) Trim columns
# ----------------------------
df = bronze.select(
    trim(col("event_id")).alias("event_id"),
    trim(col("user_id")).alias("user_id"),
    trim(col("content_id")).alias("content_id"),
    trim(col("start_time")).alias("start_time_str"),
    trim(col("end_time")).alias("end_time_str"),
    trim(col("device")).alias("device"),
    trim(col("geo")).alias("geo"),
    trim(col("bitrate")).alias("bitrate_str"),
    trim(col("errors")).alias("errors")
)

# ----------------------------
# 2) Drop unusable rows (invalid IDs)
# ----------------------------
df = df.filter(
    ~(col("event_id").isNull() | lower(col("event_id")).startswith("invalid")) &
    ~(col("user_id").isNull() | lower(col("user_id")).startswith("invalid")) &
    ~(col("content_id").isNull() | lower(col("content_id")).startswith("invalid"))
)

# ----------------------------
# 3) Parse timestamps, drop rows with both missing
# ----------------------------
df = df.withColumn(
    "start_time",
    coalesce(
        to_timestamp(col("start_time_str"), "yyyy-MM-dd HH:mm:ss"),
        to_timestamp(col("start_time_str"), "yyyy/MM/dd HH:mm:ss"),
        to_timestamp(col("start_time_str"))
    )
).withColumn(
    "end_time",
    coalesce(
        to_timestamp(col("end_time_str"), "yyyy-MM-dd HH:mm:ss"),
        to_timestamp(col("end_time_str"), "yyyy/MM/dd HH:mm:ss"),
        to_timestamp(col("end_time_str"))
    )
).drop("start_time_str", "end_time_str")

# Drop rows with no valid time at all
df = df.filter(~(col("start_time").isNull() & col("end_time").isNull()))
df = df.filter(~(col("start_time").isNull() ))

# ----------------------------
# 4) Handle crashed sessions (start_time valid, end_time null, errors > 0)
# ----------------------------
df = df.withColumn(
    "session_status",
    when((col("start_time").isNotNull()) & (col("end_time").isNull()) & (col("errors") > 0),
         lit("crashed"))
    .otherwise(lit("completed"))
)

df = df.withColumn(
    "end_time",
    when((col("session_status") == "crashed"),
         (col("start_time").cast("long") + lit(30)).cast("timestamp"))  # impute 30s
    .otherwise(col("end_time"))
)

# ----------------------------
# 5) Standardize categorical fields
# ----------------------------
df = df.withColumn(
    "device",
    when(col("device").isNull() | (col("device") == "") | lower(col("device")).startswith("invalid"),
         "unknown").otherwise(col("device"))
).withColumn(
    "geo",
    when(col("geo").isNull() | (col("geo") == "") | lower(col("geo")).startswith("invalid"),
         "unknown").otherwise(col("geo"))
)

# ----------------------------
# 6) Standardize bitrate
# ----------------------------
df = df.withColumn("bitrate_int", col("bitrate_str").cast("int"))
df = df.withColumn(
    "bitrate",
    when(col("bitrate_int").isin(ALLOWED_BITRATES), col("bitrate_int")).otherwise(lit("unknown"))
).drop("bitrate_str", "bitrate_int")

# ---------------
# 7) Standardize errors (string categories)
# ----------------------------
VALID_ERRORS = ["network_drop", "buffering", "crash"]

df = df.withColumn(
    "errors",
    when(col("errors").isNull() | (trim(col("errors")) == ""), lit("no_error"))  # null/blank → no_error
    .when(lower(col("errors")).isin([e.lower() for e in VALID_ERRORS]), lower(col("errors")))  # keep valid
    .when(lower(col("errors")).like("invalid%"), lit("crash"))  # invalidX → crash
    .otherwise(lit("crash"))  # everything else → crash
)


# ----------------------------
# 8) Optional: compute session duration
# ----------------------------
df = df.withColumn(
    "watch_span",
    when((col("start_time").isNotNull()) & (col("end_time").isNotNull()),
         col("end_time").cast("long") - col("start_time").cast("long"))
)

# ----------------------------
# 9) Save to Silver
# ----------------------------
df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalog}.silver.silver_events")


In [0]:
%sql
select * from hetul_catalog_ott_02.silver.silver_events

event_id,user_id,content_id,device,geo,errors,start_time,end_time,session_status,bitrate,watch_span
E000002,U00027,C0035,Web,UK,network_drop,2025-08-28T23:20:07.064571Z,2025-08-29T00:48:07.064571Z,completed,unknown,5280
E000003,U00030,C0044,Mobile,DE,buffering,2025-08-29T02:03:07.064571Z,2025-08-29T02:25:07.064571Z,completed,240,1320
E000004,U00212,C0059,Tablet,IN,crash,2025-08-28T19:21:07.064571Z,2025-08-28T19:30:07.064571Z,completed,unknown,540
E000005,U00306,C0080,Web,DE,no_error,2025-08-28T15:07:07.064571Z,2025-08-28T16:06:07.064571Z,completed,1080,3540
E000006,U00442,C0002,Tablet,DE,crash,2025-08-29T04:55:07.064571Z,2025-08-29T05:10:07.064571Z,completed,2160,900
E000007,U00312,C0059,Web,DE,network_drop,2025-08-29T03:53:07.064571Z,2025-08-29T04:04:07.064571Z,completed,360,660
E000010,U00338,C0048,Tablet,JP,network_drop,2025-08-28T17:28:07.064571Z,2025-08-28T18:22:07.064571Z,completed,1080,3240
E000011,U00252,C0085,SmartTV,IN,buffering,2025-08-28T15:02:07.064571Z,2025-08-28T17:40:07.064571Z,completed,unknown,9480
E000012,U00324,C0039,Tablet,DE,no_error,2025-08-28T11:24:07.064571Z,2025-08-28T12:37:07.064571Z,completed,480,4380
E000013,U00471,C0021,Tablet,JP,crash,2025-08-29T03:46:07.064571Z,2025-08-29T04:08:07.064571Z,completed,480,1320


In [0]:
from pyspark.sql import functions as F

catalog = "hetul_catalog_ott_02"

# Load Silver Events
events = spark.table(f"{catalog}.silver.silver_events")


For the Operation Dashboard


In [0]:
from pyspark.sql import functions as F

gold_concurrent_viewers = (
    events
    .withWatermark("start_time", "5 minutes")
    .groupBy(
        F.window("start_time", "5 minutes"),
        "content_id"
    )
    .agg(
        F.countDistinct("user_id").alias("concurrent_viewers")
    )
    .withColumn("window_start", F.col("window").start)
    .withColumn("window_end", F.col("window").end)
    .drop("window")
)

gold_concurrent_viewers.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalog}.gold.gold_concurrent_viewers")


In [0]:
gold_stream_quality_fact = (
    events
    .withColumn("day", F.to_date("start_time"))
    .groupBy("day", "content_id", "geo", "device")
    .agg(
        F.avg("bitrate").alias("avg_bitrate"),
        F.sum(F.when(F.col("errors") == "buffering", 1).otherwise(0)).alias("buffering_count"),
        F.sum(F.when(F.col("errors") == "crash", 1).otherwise(0)).alias("crash_count"),
        F.sum(F.when(F.col("errors") == "network_drop", 1).otherwise(0)).alias("network_drop_count")
    )
)

gold_stream_quality_fact.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.gold.gold_stream_quality_fact")


In [0]:
from pyspark.sql import functions as F

gold_device_usage = (
    # Start with the silver layer table
    spark.table(f"{catalog}.silver.silver_events")
    
    # 1. Add the 'day' column from the start_time
    .withColumn("day", F.to_date("start_time"))
    
    # Add a filter to drop records where 'day' is null.
    .filter(F.col("day").isNotNull())
    
    # 2. Clean the 'device' column
    .withColumn("device", F.lower(F.col("device")))
    .withColumn(
        "device",
        F.when(
            F.lower(F.col("device")).isin("invalid", "n/a", "unknown", "null"),
            "unknown"
        ).otherwise(F.col("device"))
    )
    
    # 3. Calculate 'watch_time_sec' only for valid sessions
    .withColumn(
        "watch_time_sec",
        F.when(
            F.col("end_time").isNotNull() & (F.col("end_time") >= F.col("start_time")),
            F.col("end_time").cast("long") - F.col("start_time").cast("long")
        )
        .otherwise(F.lit(0))
    )
    
    # 4. Group by day and device to perform aggregations
    .groupBy("day", "device")
    .agg(
        F.countDistinct("user_id").alias("active_users"),
        F.sum("watch_time_sec").alias("total_watch_time_sec")
    )
)

gold_device_usage.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.gold.gold_device_usage")

In [0]:
gold_geo_usage = (
    events
    .withColumn("day", F.to_date("start_time"))
    .withColumn("watch_time_sec", F.col("end_time").cast("long") - F.col("start_time").cast("long"))
    .groupBy("day", "geo")
    .agg(
        F.countDistinct("user_id").alias("active_users"),
        F.sum("watch_time_sec").alias("total_watch_time_sec"),
        F.count("*").alias("total_sessions")
    )
)

gold_geo_usage.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.gold.gold_geo_usage")


In [0]:
from pyspark.sql import functions as F

# The correct approach is to start with a clean, standardized Silver table.
silver_events = spark.table(f"{catalog}.silver.silver_events")

gold_error_fact = (
    silver_events
    # 1. Derive the 'day' column from the clean 'start_time' and drop nulls.
    .withColumn("day", F.to_date("start_time"))
    .filter(F.col("day").isNotNull())
    
    # 2. Filter out records that are not errors before aggregation.
    # We use a filter to exclude any value that indicates 'no error'
    .filter(~F.lower(F.col("errors")).isin("no_error", "none"))

    # 3. Group by the now clean dimensions.
    .groupBy("day", "errors", "geo", "device")
    .agg(
        F.count("*").alias("error_occurrences"),
        F.countDistinct("user_id").alias("unique_users_affected")
    )
)

gold_error_fact.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalog}.gold.gold_error_fact")

Product Dashboard

In [0]:
from pyspark.sql import functions as F

# ----------------------------
# Load Silver Tables
# ----------------------------
users   = spark.table(f"{catalog}.silver.silver_users")
content = spark.table(f"{catalog}.silver.silver_content")
events  = spark.table(f"{catalog}.silver.silver_events")

# Add useful derived columns
events = (
    events
    .withColumn("day", F.to_date("start_time"))
    .withColumn("watch_minutes", F.col("watch_span")/60.0)  # convert seconds → minutes
)

# ==============================================================
# 1. gold_content_dim → Content dimension
# ==============================================================

gold_content_dim = (
    content
    .select(
        "content_id",
        "title",
        "genre",
        "duration_min",
        "region_availability"
    )
    .dropDuplicates(["content_id"])
    .withColumnRenamed("duration_min", "duration")
    .withColumnRenamed("region_availability", "region")
)

gold_content_dim.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalog}.gold.gold_content_dim")


# ==============================================================
# 2. gold_content_engagement_fact → Engagement metrics
# ==============================================================

gold_content_engagement_fact = (
    events
    .join(content, "content_id", "left")
    .groupBy("content_id", "region_availability", "day")
    .agg(
        F.sum("watch_minutes").alias("total_watch_hours"),
        F.avg("watch_minutes").alias("avg_watch_minutes"),
        F.countDistinct("user_id").alias("unique_viewers")
    )
    .withColumn("total_watch_hours", F.col("total_watch_hours")/60.0)  # minutes → hours
    .withColumnRenamed("region_availability", "region")
)

gold_content_engagement_fact.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalog}.gold.gold_content_engagement_fact")


# ==============================================================
# 3. gold_genre_trends → Genre-level watch trends
# ==============================================================

gold_genre_trends = (
    events
    .join(content, "content_id", "left")
    .groupBy("genre", "day")
    .agg(
        F.sum("watch_minutes").alias("total_watch_hours"),
        F.countDistinct("user_id").alias("unique_viewers")
    )
    .withColumn("total_watch_hours", F.col("total_watch_hours")/60.0)
)

gold_genre_trends.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalog}.gold.gold_genre_trends")


# ==============================================================
# 4. gold_user_retention_fact → D1/D7/D30 retention
# ==============================================================

first_watch = (
    events.groupBy("user_id")
    .agg(F.min("day").alias("signup_day"))
)

retention = (
    events.join(first_watch, "user_id")
    .withColumn("days_since_signup", F.datediff("day", "signup_day"))
)

gold_user_retention_fact = (
    retention
    .join(content, "content_id", "left")
    .groupBy("genre", "days_since_signup")
    .agg(F.countDistinct("user_id").alias("retained_users"))
    .filter(F.col("days_since_signup").isin([1, 7, 30]))
)

gold_user_retention_fact.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalog}.gold.gold_user_retention_fact")


# ==============================================================
# 5. gold_cross_device_behavior → User device usage
# ==============================================================

gold_cross_device_behavior = (
    events
    .groupBy("user_id", "day")
    .agg(F.collect_set("device").alias("devices_used"))
    .withColumn("num_devices", F.size("devices_used"))
)

gold_cross_device_behavior.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalog}.gold.gold_cross_device_behavior")


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, countDistinct, when, lit

catalog = "hetul_catalog_ott_02"

# Load Silver tables
users   = spark.table(f"{catalog}.silver.silver_users")
events  = spark.table(f"{catalog}.silver.silver_events")
ads     = spark.table(f"{catalog}.silver.silver_ads")
content = spark.table(f"{catalog}.silver.silver_content")

# ==========================================================
# 1. gold_user_dim
# ==========================================================
# Take latest region per user (from events.geo)
user_region = (events
    .groupBy("user_id")
    .agg(F.first("geo", ignorenulls=True).alias("region"))
)

gold_user_dim = (users
    .join(user_region, "user_id", "left")
    .select("user_id", "plan_type", "billing_cycle", "churn_flag", "region")
)

gold_user_dim.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalog}.gold.gold_user_dim")

# ==========================================================
# 2. gold_subscription_fact
# ==========================================================
gold_subscription_fact = (users
    .agg(
        F.count(when(col("payment_status") == "active", 1)).alias("active_subs"),
        F.count(when(col("churn_flag") == 1, 1)).alias("churned_users"),
        F.count(when(col("plan_type") == "premium", 1)).alias("premium_users"),
        F.count(when(col("plan_type") == "basic", 1)).alias("basic_users"),
        F.count(when(col("plan_type") == "standard", 1)).alias("standard_users")
    )
)

gold_subscription_fact.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalog}.gold.gold_subscription_fact")

# ==========================================================
# 3. gold_churn_features (ML-ready)
# ==========================================================
watch_features = (events
    .groupBy("user_id")
    .agg(
        F.avg("watch_span").alias("avg_watch_time"),
        F.sum("watch_span").alias("total_watch_time"),
        F.count("*").alias("session_count"),
        F.count(when(col("errors") != "no_error", 1)).alias("error_count"),
        countDistinct("device").alias("distinct_devices")
    )
)

genre_features = (events
    .join(content, "content_id", "left")
    .groupBy("user_id")
    .agg(countDistinct("genre").alias("distinct_genres_watched"))
)

gold_churn_features = (users
    .join(watch_features, "user_id", "left")
    .join(genre_features, "user_id", "left")
    .fillna(0, ["avg_watch_time","total_watch_time","session_count","error_count","distinct_devices","distinct_genres_watched"])
    .select("user_id", "plan_type", "billing_cycle", "payment_status", 
            "avg_watch_time", "total_watch_time", "session_count", "error_count", 
            "distinct_devices", "distinct_genres_watched", "churn_flag")
)

gold_churn_features.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalog}.gold.gold_churn_features")

# ==========================================================
# 4. gold_ad_campaign_fact
# ==========================================================
gold_ad_campaign_fact = (ads
    .groupBy("campaign", "content_id")
    .agg(
        F.sum("impressions").alias("total_impressions"),
        F.sum("clicks").alias("total_clicks"),
        (F.sum("clicks") / F.sum("impressions")).alias("CTR"),
        F.sum("revenue").alias("total_revenue")
    )
)

gold_ad_campaign_fact.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalog}.gold.gold_ad_campaign_fact")

# ==========================================================
# 5. gold_ad_attribution_fact
# ==========================================================
gold_ad_attribution_fact = (ads
    .join(events, "content_id", "inner")
    .groupBy("ad_id", "campaign", "content_id", "user_id")
    .agg(
        F.sum("impressions").alias("impressions"),
        F.sum("clicks").alias("clicks"),
        F.sum("revenue").alias("revenue"),
        F.sum("watch_span").alias("watch_time_after_ad")
    )
)

gold_ad_attribution_fact.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalog}.gold.gold_ad_attribution_fact")

# ==========================================================
# 6. gold_marketing_funnel
# ==========================================================
gold_marketing_funnel = (users
    .agg(
        F.count("*").alias("total_users"),
        F.count(when(col("payment_status") == "active", 1)).alias("active_users"),
        F.count(when((col("payment_status") == "active") & (col("plan_type") != "unknown"), 1)).alias("paying_users"),
        F.count(when(col("churn_flag") == 1, 1)).alias("churned_users")
    )
    .withColumn("trial_users", lit(0))  # Placeholder, if you add trial flag later
)

gold_marketing_funnel.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalog}.gold.gold_marketing_funnel")


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, lit

catalog = "hetul_catalog_ott_02"

# Load Silver tables
users   = spark.table(f"{catalog}.silver.silver_users")
ads     = spark.table(f"{catalog}.silver.silver_ads")
events  = spark.table(f"{catalog}.silver.silver_events")

# ==========================================================
# 1. gold_revenue_fact
# ==========================================================

# --- Subscription revenue (assumption: ARPU mapping by plan) ---
plan_revenue_map = {
    "basic": 5.0,
    "standard": 10.0,
    "premium": 15.0,
    "unknown": 0.0
}
plan_revenue_expr = F.create_map([lit(x) for x in sum(plan_revenue_map.items(), ())])

subscription_revenue = (users
    .withColumn("subscription_revenue", plan_revenue_expr[col("plan_type")])
    .withColumn("date", F.current_date())  # Replace with billing date if available
    .groupBy("date", "plan_type")
    .agg(F.sum("subscription_revenue").alias("total_subscription_revenue"))
)

# --- Ad revenue ---
ad_revenue = (ads
    .withColumn("date", F.current_date())  # Replace with ad_date if available
    .groupBy("date", "content_id")
    .agg(F.sum("revenue").alias("total_ad_revenue"))
)

# Combine both
gold_revenue_fact = (subscription_revenue
    .join(ad_revenue, "date", "outer")
    .fillna(0, ["total_subscription_revenue", "total_ad_revenue"])
    .withColumn("total_revenue", col("total_subscription_revenue") + col("total_ad_revenue"))
)

gold_revenue_fact.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalog}.gold.gold_revenue_fact")

# ==========================================================
# 2. gold_user_finance_fact
# ==========================================================
# ----------------------------------------------------------
# 2. gold_user_finance_fact (fixed)
# ----------------------------------------------------------

# Subscription revenue per user
user_subs = (users
    .withColumn("subscription_revenue", plan_revenue_expr[col("plan_type")])
    .select("user_id", "billing_cycle", "subscription_revenue")
)

# Map users → content from events
user_content_map = (events
    .select("user_id", "content_id")
    .distinct()
)

# Join to ads via content_id
user_ads = (user_content_map
    .join(ads.groupBy("content_id")
              .agg(F.sum("revenue").alias("ad_revenue")),
          "content_id", "left")
    .groupBy("user_id")
    .agg(F.sum("ad_revenue").alias("ad_revenue"))
)

# Combine subscription + ad revenue
gold_user_finance_fact = (user_subs
    .join(user_ads, "user_id", "left")
    .fillna(0, ["ad_revenue"])
    .withColumn("total_revenue", col("subscription_revenue") + col("ad_revenue"))
    .groupBy("user_id", "billing_cycle")
    .agg(
        F.first("subscription_revenue").alias("subscription_revenue"),
        F.first("ad_revenue").alias("ad_revenue"),
        F.sum("total_revenue").alias("ltv"),  # lifetime value
        (F.sum("total_revenue")/F.count("user_id")).alias("arpu")
    )
)

gold_user_finance_fact.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalog}.gold.gold_user_finance_fact")


# ==========================================================
# 3. gold_campaign_roi
# ==========================================================
# Assume campaign_spend not available → placeholder
gold_campaign_roi = (ads
    .groupBy("campaign")
    .agg(
        F.sum("revenue").alias("campaign_revenue"),
        F.lit(1000.0).alias("campaign_spend")  # placeholder
    )
    .withColumn("roi", col("campaign_revenue") / col("campaign_spend"))
)

gold_campaign_roi.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalog}.gold.gold_campaign_roi")

# ==========================================================
# 4. gold_subscription_fact (already exists, reused)
# ==========================================================
# You already built this in marketing → no need to rewrite
